<h1>The Figure Data Structure in Python</h1>
The structure of a figure - data, traces and layout explained.

## <h3 id="overview">Overview</h3>

<p>The <code>plotly</code> Python package exists to <a href="https://plotly.com/python/creating-and-updating-figures/">create, manipulate</a> and <a href="https://plotly.com/python/renderers/">render</a> graphical figures (i.e. charts, plots, maps and diagrams) represented by data structures also referred to as figures.
The rendering process uses the <a href="https://plotly.com/javascript/">Plotly.js JavaScript library</a> under the hood although Python developers using this module very rarely need to interact with the Javascript library directly, if ever.
Figures can be represented in Python either as dicts or as instances of the <code>plotly.graph_objects.Figure</code> class, and are serialized as text in <a href="https://json.org/">JavaScript Object Notation (JSON)</a> before being passed to Plotly.js.</p>

<blockquote><p>Note: the recommended entry-point into the plotly package is the <a href="/python/plotly-express/">high-level plotly.express module, also known as Plotly Express</a>, which consists of Python functions which return fully-populated <code>plotly.graph_objects.Figure</code> objects. This page exists to document the architecture of the data structure that these objects represent, for users who wish to understand more about how to customize them, or assemble them from <a href="/python/graph-objects/">other <code>plotly.graph_objects</code> components</a>.</p></blockquote>

<p>Viewing the underlying data structure for any <code>plotly.graph_objects.Figure</code> object, including those returned by Plotly Express, can be done via <code>print(fig)</code> or, in JupyterLab, with the special <code>fig.show("json")</code> renderer. Figures also support <code>fig.to_dict()</code> and <code>fig.to_json()</code> methods. <code>print()</code>ing the figure will result in the often-verbose <code>layout.template</code> key being represented as ellipses <code>'...'</code> for brevity.</p>

In [ ]:
import plotly.express as px

fig = px.line(x=["a","b","c"], y=[1,3,2], title="sample figure")
print(fig)
fig.show()

Figure({
    'data': [{'hovertemplate': 'x=%{x}<br>y=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': array(['a', 'b', 'c'], dtype=object),
              'xaxis': 'x',
              'y': array([1, 3, 2]),
              'yaxis': 'y'}],
    'layout': {'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'sample figure'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'x'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'y'}}}
})


## <h3 id="figures-as-trees-of-attributes">Figures as Trees of Attributes</h3>

<p>Plotly.js supports inputs adhering to a well-defined schema, whose overall architecture is explained in this page and which is exhaustively documented in the <a href="https://plotly.com/python/reference/index/">Figure Reference</a> (which is itself generated from a <a href="https://raw.githubusercontent.com/plotly/plotly.js/master/dist/plot-schema.json">machine-readable JSON representation of the schema</a>). Figures are represented as trees with named nodes called "attributes". The root node of the tree has three top-level attributes: <code>data</code>, <code>layout</code> and <code>frames</code> (see below).</p>

<p>Attributes are referred to in text and in the <a href="https://plotly.com/python/reference/index/">Figure Reference</a> by their full "path" i.e. the dot-delimited concatenation of their parents. For example <code>"layout.width"</code> refers to the attribute whose key is <code>"width"</code> inside a dict which is the value associated with a key <code>"layout"</code> at the root of the figure. If one of the parents is a list rather than a dict, a set of brackets is inserted in the path when referring to the attribute in the abstract, e.g. <code>"layout.annotations[].text"</code>. Finally, as explained below, the top-level "data" attribute defines a list of typed objects called "traces" with the schema dependent upon the type, and these attributes' paths are listed in the <a href="https://plotly.com/python/reference/index/">Figure Reference</a> as <code>"data[type=scatter].name"</code>.</p>

<p>The <a href="https://plotly.com/python/graph-objects/"><code>plotly.graph_objects</code> module contains an automatically-generated hierarchy of  Python classes</a> which represent non-leaf attributes in the figure schema and provide a Pythonic API for them. When <a href="https://plotly.com/python/creating-and-updating-figures/">manipulating a <code>plotly.graph_objects.Figure</code> object</a>, attributes can be set either directly using Python object attributes e.g. <code>fig.layout.title.font.family="Open Sans"</code> or using <a href="https://plotly.com/python/creating-and-updating-figures/#magic-underscore-notation">update methods and "magic underscores"</a> e.g. <code>fig.update_layout(title_font_family="Open Sans")</code></p>

<p>When building a figure, it is <em>not necessary to populate every attribute</em> of every object. At render-time, <a href="https://plotly.com/python/figure-introspection/">the JavaScript layer will compute default values</a> for each required unspecified attribute, depending upon the ones that are specified, as documented in the <a href="https://plotly.com/python/reference/index/">Figure Reference</a>. An example of this would be <code>layout.xaxis.range</code>, which may be specified explicitly, but if not will be computed based on the range of <code>x</code> values for every trace linked to that axis. The JavaScript layer will ignore unknown attributes or malformed values, although the <code>plotly.graph_objects</code> module provides Python-side validation for attribute values. Note also that if <a href="https://plotly.com/python/templates/">the <code>layout.template</code> key is present (as it is by default)</a> then default values will be drawn first from the contents of the template and only if missing from there will the JavaScript layer infer further defaults. The built-in template can be disabled by setting <code>layout.template="none"</code>.</p>

## <h3 id="the-toplevel-data-attribute">The Top-Level <code>data</code> Attribute</h3>

<p>The first of the three top-level attributes of a figure is <code>data</code>, whose value must be a list of dicts referred to as "traces".</p>

<ul>

<li>Each trace has one of more than 40 possible types (see below for a list organized by subplot type, including e.g. <a href="https://plotly.com/python/line-and-scatter/"><code>scatter</code></a>, <a href="https://plotly.com/python/bar-charts/"><code>bar</code></a>, <a href="https://plotly.com/python/pie-charts/"><code>pie</code></a>, <a href="https://plotly.com/python/3d-surface-plots/"><code>surface</code></a>, <a href="https://plotly.com/python/choropleth-maps/"><code>choropleth</code></a> etc), and represents a set of related graphical marks in a figure. Each trace must have a <code>type</code> attribute which defines the other allowable attributes.</li>

<li>Each trace is drawn on a single <a href="https://plotly.com/python/subplots/">subplot</a> whose type must be compatible with the trace's type, or is its own subplot (see below).</li>

<li>Traces may have a single <a href="https://plotly.com/python/legend/">legend</a> entry, with the exception of pie and funnelarea traces (see below).</li>

<li>Certain trace types support <a href="https://plotly.com/python/colorscales/">continuous color, with an associated colorbar</a>, which can be controlled by attributes either within the trace, or within the layout when using the <a href="https://plotly.com/python/colorscales/">coloraxis attribute</a>.</li>

</ul>

## <h3 id="the-toplevel-layout-attribute">The Top-Level <code>layout</code> Attribute</h3>

<p>The second of the three top-level attributes of a figure is <code>layout</code>, whose value is referred to in text as "the layout" and must be a dict, containing attributes that control positioning and configuration of non-data-related parts of the figure such as:</p>

<ul>

<li>Dimensions and margins, which define the bounds of "paper coordinates" (see below)</li>

<li>Figure-wide defaults: <a href="https://plotly.com/python/templates/">templates</a>, <a href="https://plotly.com/python/figure-labels/">fonts</a>, colors, hover-label and modebar defaults</li>

<li><a href="https://plotly.com/python/figure-labels/">Title</a> and <a href="https://plotly.com/python/legend/">legend</a> (positionable in container and/or paper coordinates)</li>

<li><a href="https://plotly.com/python/colorscales/">Color axes and associated color bars</a> (positionable in paper coordinates)</li>

<li>Subplots of various types on which can be drawn multiple traces and which are positioned in paper coordinates:<ul>
<li><code>xaxis</code>, <code>yaxis</code>, <code>xaxis2</code>, <code>yaxis3</code> etc: X and Y cartesian axes, the intersections of which are cartesian subplots</li>
<li><code>scene</code>, <code>scene2</code>, <code>scene3</code> etc: 3d scene subplots</li>
<li><code>ternary</code>, <code>ternary2</code>, <code>ternary3</code>, <code>polar</code>, <code>polar2</code>, <code>polar3</code>, <code>geo</code>, <code>geo2</code>, <code>geo3</code>, <code>map</code>, <code>map2</code>, <code>map3</code>, <code>smith</code>, <code>smith2</code> etc: ternary, polar, geo, map or smith subplots</li>
</ul>
</li>

<li>Non-data marks which can be positioned in paper coordinates, or in data coordinates linked to 2d cartesian subplots:<ul>
<li><code>annotations</code>: <a href="https://plotly.com/python/text-and-annotations/">textual annotations with or without arrows</a></li>
<li><code>shapes</code>: <a href="https://plotly.com/python/shapes/">lines, rectangles, ellipses or open or closed paths</a></li>
<li><code>images</code>: <a href="https://plotly.com/python/images/">background or decorative images</a></li>
</ul>
</li>

<li>Controls which can be positioned in paper coordinates and which can trigger Plotly.js functions when interacted with by a user:<ul>
<li><code>updatemenus</code>: <a href="https://plotly.com/python/custom-buttons/">single buttons, toggles</a> and <a href="https://plotly.com/python/dropdowns/">dropdown menus</a></li>
<li><code>sliders</code>: <a href="https://plotly.com/python/sliders/">slider controls</a></li>
</ul>
</li>

</ul>

## <h3 id="the-toplevel-frames-attribute">The Top-Level <code>frames</code> Attribute</h3>

<p>The third of the three top-level attributes of a figure is <code>frames</code>, whose value must be a list of dicts that define sequential frames in an <a href="https://plotly.com/python/animations/">animated plot</a>. Each frame contains its own data attribute as well as other parameters. Animations are usually triggered and controlled via controls defined in layout.sliders and/or layout.updatemenus</p>

## <h3 id="the-config-object">The <code>config</code> Object</h3>

<p>At <a href="https://plotly.com/python/renderers/">render-time</a>, it is also possible to control certain figure behaviors which are not considered part of the figure proper i.e. the behavior of the "modebar" and how the figure relates to mouse actions like scrolling etc. The object that contains these options is called the <a href="https://plotly.com/python/configuration-options/"><code>config</code>, and has its own documentation page</a>. It is exposed in Python as the <code>config</code> keyword argument of the <code>.show()</code> method on <code>plotly.graph_objects.Figure</code> objects.</p>

## <h3 id="positioning-with-paper-container-coordinates-or-axis-domain-coordinates">Positioning With Paper, Container Coordinates, or Axis Domain Coordinates</h3>

<p>Various figure components configured within the layout of the figure support positioning attributes named <code>x</code> or <code>y</code>, whose values may be specified in "paper coordinates" (sometimes referred to as "plot fractions" or "normalized coordinates"). Examples include <code>layout.xaxis.domain</code> or <code>layout.legend.x</code> or <code>layout.annotation[].x</code>.</p>

<p>Positioning in paper coordinates is <em>not</em> done in absolute pixel terms, but rather in terms relative to a coordinate system defined with an origin <code>(0,0)</code> at <code>(layout.margin.l, layout.margin.b)</code> and a point <code>(1,1)</code> at <code>(layout.width-layout.margin.r, layout.height-layout.margin.t)</code> (note: <code>layout.margin</code> values are pixel values, as are <code>layout.width</code> and <code>layout.height</code>). Paper coordinate values less than 0 or greater than 1 are permitted, and refer to areas within the plot margins.</p>

<p>To position an object in "paper" coordinates, the corresponding axis reference is set to <code>"paper"</code>. For instance a shape's <code>xref</code> attribute would be set to <code>"paper"</code> so that the <code>x</code> value of the shape refers to its position in paper coordinates.</p>

<p>Note that the contents of the <code>layout.margin</code> attribute are by default computed based on the position and dimensions of certain items like the title or legend, and may be made dependent on the position and dimensions of tick labels as well when setting the <code>layout.xaxis.automargin</code> attribute to <code>True</code>. This has the effect of automatically increasing the margin values and therefore shrinking the physical area defined between the <code>(0,0)</code> and <code>(1,1)</code> points. Positioning certain items at paper coordinates less than 0 or greater than 1 will also trigger this behavior. The <code>layout.width</code> and <code>layout.height</code>, however, are taken as givens, so a figure will never grow or shrink based on its contents.</p>

<p>The figure title may be positioned using "container coordinates" which have <code>(0,0)</code> and <code>(1,1)</code> anchored at the bottom-left and top-right of the figure, respectively, and therefore are independent of the values of layout.margin.</p>

<p>Furthermore, shapes, annotations, and images can be placed relative to an axis's domain so that, for instance, an <code>x</code> value of <code>0.5</code> would place the object halfway along the x-axis, regardless of the domain as specified in the <code>layout.xaxis.domain</code> attribute. This behavior can be specified by adding <code>' domain'</code> to the axis reference in the axis referencing attribute of the object.
For example, setting <code>yref = 'y2 domain'</code> for a shape will refer to the length and position of the axis named <code>y2</code>.</p>

## <h3 id="2d-cartesian-trace-types-and-subplots">2D Cartesian Trace Types and Subplots</h3>

<p>The most commonly-used kind of subplot is a <a href="https://plotly.com/python/axes/">two-dimensional Cartesian subplot</a>. Traces compatible with these subplots support <code>xaxis</code> and <code>yaxis</code> attributes whose values must refer to corresponding objects in the layout portion of the figure. For example, if <code>xaxis="x"</code>, and <code>yaxis="y"</code> (which is the default) then this trace is drawn on the subplot at the intersection of the axes configured under <code>layout.xaxis</code> and <code>layout.yaxis</code>, but if <code>xaxis="x2"</code> and <code>yaxis="y3"</code> then the trace is drawn at the intersection of the axes configured under <code>layout.xaxis2</code> and <code>layout.yaxis3</code>. Note that attributes such as <code>layout.xaxis</code> and <code>layout.xaxis2</code> etc do not have to be explicitly defined, in which case default values will be inferred. Multiple traces of different types can be drawn on the same subplot.</p>

<p>X- and Y-axes support the <code>type</code> attribute, which enables them to represent <a href="https://plotly.com/python/axes/">continuous values (<code>type="linear"</code>, <code>type="log"</code>)</a>, <a href="https://plotly.com/python/time-series/">temporal values (<code>type="date"</code>)</a> or <a href="https://plotly.com/python/bar-charts/">categorical values (<code>type="category"</code>, <code>type="multicategory</code>)</a>. Axes can also be overlaid on top of one another to create <a href="https://plotly.com/python/multiple-axes/">dual-axis or multiple-axis charts</a>. 2-d cartesian subplots lend themselves very well to creating <a href="https://plotly.com/python/facet-plots/">"small multiples" figures, also known as facet or trellis plots</a>.</p>

<p>The following trace types are compatible with 2d-cartesian subplots via the <code>xaxis</code> and <code>yaxis</code> attributes:</p>

<ul>

<li>scatter-like trace types: <a href="https://plotly.com/python/line-and-scatter/"><code>scatter</code></a> and <a href="https://plotly.com/python/webgl-vs-svg/"><code>scattergl</code></a>, which can be used to draw <a href="https://plotly.com/python/line-and-scatter/">scatter plots</a>, <a href="https://plotly.com/python/line-charts/">line plots and curves</a>, <a href="https://plotly.com/python/time-series/">time-series plots</a>, <a href="https://plotly.com/python/bubble-charts/">bubble charts</a>, <a href="https://plotly.com/python/dot-plots/">dot plots</a> and <a href="https://plotly.com/python/filled-area-plots/">filled areas</a> and also support <a href="https://plotly.com/python/error-bars/">error bars</a></li>

<li><a href="https://plotly.com/python/bar-charts/"><code>bar</code></a>, <a href="https://plotly.com/python/funnel-charts/"><code>funnel</code></a>, <a href="https://plotly.com/python/waterfall-charts/"><code>waterfall</code></a>: bar-like trace types which can also be used to draw <a href="https://plotly.com/python/gantt/">timelines and Gantt charts</a></li>

<li><a href="https://plotly.com/python/histograms/"><code>histogram</code></a>: an <em>aggregating</em> bar-like trace type</li>

<li><a href="https://plotly.com/python/box-plots/"><code>box</code></a> and <a href="https://plotly.com/python/box-plots/"><code>violin</code></a>: 1-dimensional distribution-like trace types</li>

<li><a href="https://plotly.com/python/2D-Histogram/"><code>histogram2d</code></a> and <a href="/python/2d-histogram-contour/"><code>histogram2dcontour</code></a>: 2-dimensional distribution-like density trace types</li>

<li><a href="https://plotly.com/python/imshow/"><code>image</code></a>, <a href="https://plotly.com/python/heatmaps/"><code>heatmap</code></a> and <a href="https://plotly.com/python/contour-plots/"><code>contour</code></a>: matrix trace types</li>

<li><a href="https://plotly.com/python/ohlc-charts/"><code>ohlc</code></a> and <a href="https://plotly.com/python/candlestick-charts/"><code>candlestick</code></a>: stock-like trace types</li>

<li><a href="https://plotly.com/python/carpet-plot/"><code>carpet</code></a>: a special trace type for building <a href="https://plotly.com/python/carpet-plot/">carpet plots</a>, in that other traces can use as subplots (see below)</li>

<li><a href="https://plotly.com/python/splom/"><code>splom</code></a>: multi-dimensional scatter plots which implicitly refer to many 2-d cartesian subplots at once.</li>

</ul>

## <h3 id="3d-polar-ternary-and-smith-trace-types-and-subplots">3D, Polar, Ternary and Smith Trace Types and Subplots</h3>

<p>Beyond 2D cartesian subplots, figures can include <a href="/python/3d-charts/">three-dimensional cartesian subplots</a>, <a href="/python/polar-chart/">polar subplots</a>, <a href="/python/ternary-plots/">ternary subplots</a> and <a href="/python/smith-charts/">smith subplots</a>. The following trace types support attributes named <code>scene</code>, <code>polar</code>, <code>smith</code> or <code>ternary</code>, whose values must refer to corresponding objects in the layout portion of the figure i.e. <code>ternary="ternary2"</code> etc. Note that attributes such as <code>layout.scene</code> and <code>layout.ternary2</code> etc do not have to be explicitly defined, in which case default values will be inferred. Multiple traces of a compatible type can be placed on the same subplot.</p>

<p>The following trace types are compatible with 3D subplots via the <code>scene</code> attribute, which contains special <a href="/python/3d-camera-controls/">camera controls</a>:</p>

<ul>

<li><a href="/python/3d-scatter-plots/"><code>scatter3d</code></a>, which can be used to draw <a href="/python/3d-scatter-plots/">individual markers</a>, <a href="/python/3d-bubble-charts/">3d bubble charts</a> and <a href="/python/3d-line-plots/">lines and curves</a></li>

<li><a href="/python/3d-surface-plots/"><code>surface</code></a> and <a href="/python/3d-mesh/"><code>mesh</code></a>: 3d surface trace types</li>

<li><a href="/python/cone-plot/"><code>cone</code></a> and <a href="/python/streamtube-plot/"><code>streamtube</code></a>: 3d vector field trace types</li>

<li><a href="/python/3d-volume-plots/"><code>volume</code></a> and <a href="/python/3d-isosurface-plots/"><code>isosurface</code></a>: 3d volume trace types</li>

</ul>

<p>The following trace types are compatible with polar subplots via the <code>polar</code> attribute:</p>

<ul>

<li>scatter-like trace types: <a href="/python/polar-chart/"><code>scatterpolar</code> and <code>scatterpolargl</code></a>, which can be used to draw individual markers, <a href="/python/radar-chart/">curves and filled areas (i.e. radar or spider charts)</a></li>

<li><a href="/python/wind-rose-charts/"><code>barpolar</code></a>: useful for <a href="/python/wind-rose-charts/">wind roses</a> and other polar bar charts</li>

</ul>

<p>The following trace types are compatible with ternary subplots via the <code>ternary</code> attribute:</p>

<ul>
<li><a href="/python/ternary-plots/"><code>scatterternary</code></a>, which can be used to draw individual markers, <a href="/python/ternary-contour/">curves and filled areas</a></li>
</ul>

<p>The following trace types are compatible with smith subplots via the <code>smith</code> attribute:</p>

<ul>
<li><a href="/python/smith-charts/"><code>scattersmith</code></a>, which can be used to draw individual markers, curves and filled areas</li>
</ul>

## <h3 id="map-trace-types-and-subplots">Map Trace Types and Subplots</h3>

<p>Figures can include two different types of map subplots: <a href="/python/map-configuration/">geo subplots for outline maps</a> and <a href="/python/tile-map-layers/">tile-based maps</a>. The following trace types support attributes named <code>geo</code> or <code>map</code>, whose values must refer to corresponding objects in the layout i.e. <code>geo="geo2"</code> etc.  Note that attributes such as <code>layout.geo2</code> and <code>layout.map</code> etc do not have to be explicitly defined, in which case default values will be inferred. Multiple traces of a compatible type can be placed on the same subplot.</p>

<p>The following trace types are compatible with geo subplots via the <code>geo</code> attribute:</p>

<ul>

<li><a href="/python/scatter-plots-on-maps/"><code>scattergeo</code></a>, which can be used to draw <a href="/python/scatter-plots-on-maps/">individual markers</a>, <a href="/python/lines-on-maps/">line and curves</a> and filled areas on outline maps</li>

<li><a href="/python/choropleth-maps/"><code>choropleth</code></a>: <a href="/python/choropleth-maps/">colored polygons</a> on outline maps</li>

</ul>

<p>The following trace types are compatible with tile map subplots via the <code>map</code> attribute:</p>

<ul>

<li><a href="/python/tile-scatter-maps/"><code>scattermap</code></a>, which can be used to draw <a href="/python/tile-scatter-maps/">individual markers</a>, <a href="/python/lines-on-tile-maps/">lines and curves</a> and <a href="/python/filled-area-tile-maps/">filled areas</a> on tile maps</li>

<li><a href="/python/tile-county-choropleth/"><code>choroplethmap</code></a>: colored polygons on tile maps</li>

<li><a href="/python/tile-density-heatmaps/"><code>densitymap</code></a>: density heatmaps on tile maps</li>

</ul>

## <h3 id="traces-which-are-their-own-subplots">Traces Which Are Their Own Subplots</h3>

<p>Certain trace types cannot share subplots, and hence have no attribute to map to a corresponding subplot in the layout. Instead, these traces are their own subplot and support a <code>domain</code> attribute for position, which enables the trace to be positioned in paper coordinates (see below). With the exception of <code>pie</code> and <code>funnelarea</code>, such traces also do not support legends (see below)</p>

<p>The following trace types are their own subplots and support a domain attribute:</p>

<ul>

<li><a href="https://plotly.com/python/pie-charts/"><code>pie</code></a> and <a href="/python/waterfall-charts/"><code>funnelarea</code></a>: one-level part-to-whole relationships with legend items</li>

<li><a href="https://plotly.com/python/sunburst-charts/"><code>sunburst</code></a> and <a href="https://plotly.com/python/treemaps/"><code>treemap</code></a>: hierarchical multi-level part-to-whole relationships</li>

<li><a href="https://plotly.com/python/parallel-coordinates-plot/"><code>parcoords</code></a> and <a href="https://plotly.com/python/parallel-categories-diagram/"><code>parcats</code></a>: continuous and categorical multidimensional figures with <a href="https://plotly.com/python/parallel-coordinates-plot/">parallel coordinates</a> and <a href="https://plotly.com/python/parallel-categories-diagram/">parallel sets</a></li>

<li><a href="https://plotly.com/python/sankey-diagram/"><code>sankey</code></a>: <a href="https://plotly.com/python/sankey-diagram/">flow diagrams</a></li>
<li><a href="https://plotly.com/python/table/"><code>table</code></a>: <a href="https://plotly.com/python/table/">text-based tables</a></li>

<li><a href="https://plotly.com/python/indicator/"><code>indicator</code></a>: big numbers, <a href="https://plotly.com/python/gauge-charts/">gauges</a>, and <a href="https://plotly.com/python/bullet-charts/">bullet charts</a></li>

</ul>

## <h3 id="carpet-trace-types-and-subplots">Carpet Trace Types and Subplots</h3>

<p>Certain trace types use <a href="https://plotly.com/python/carpet-plot/">traces of type <code>carpet</code> as a subplot</a>. These support a <code>carpet</code> attribute whose value must match the value of the <code>carpet</code> attribute of the <code>carpet</code> trace they are to be drawn on. Multiple compatible traces can be placed on the same <code>carpet</code> trace.</p>

<p>The following trace types are compatible with <code>carpet</code> trace subplots via the <code>carpet</code> attribute:</p>

<ul>

<li><a href="https://plotly.com/python/carpet-scatter/"><code>scattercarpet</code></a>, which can be used to draw individual markers, curves and filled areas</li>

<li><a href="https://plotly.com/python/carpet-plot/"><code>contourcarpet</code></a></li>

</ul>

## <h3 id="trace-types-legends-and-color-bars">Trace Types, Legends and Color Bars</h3>

<p>Traces of most types can be optionally associated with a single legend item in the <a href="https://plotly.com/python/legend/">legend</a>.
Whether or not a given trace appears in the legend is controlled via the <code>showlegend</code> attribute. Traces which are their own subplots (see above) do not support this, with the exception of traces of type <code>pie</code> and <code>funnelarea</code> for which every distinct color represented in the trace gets a separate legend item. Users may show or hide traces by clicking or double-clicking on their associated legend item. Traces that support legend items also support the <code>legendgroup</code> attribute, and all traces with the same legend group are treated the same way during click/double-click interactions.</p>

<p>The fact that legend items are linked to traces means that when using <a href="https://plotly.com/python/discrete-color/">discrete color</a>, a figure must have one trace per color in order to get a meaningful legend. <a href="https://plotly.com/python/discrete-color/">Plotly Express has robust support for discrete color</a> to make this easy.</p>

<p>Traces which support <a href="https://plotly.com/python/colorscales/">continuous color</a> can also be associated with color axes in the layout via the <code>coloraxis</code> attribute. Multiple traces can be linked to the same color axis. Color axes have a legend-like component called color bars. Alternatively, color axes can be configured within the trace itself.</p>